Making Predictions with GlassPy
===============================

**Author:** Daniel R. Cassar



## Introduction



GlassPy provides three prediction models: `GlassNet`, `VITRIFY`, and `ViscNet`.

-   `GlassNet` is a multitask deep neural network that predicts 85 glass properties and the temperature dependence of viscosity.
-   `VITRIFY` is a classification model that estimates the probability of a given composition forming a glass.
-   `ViscNet` is a physics-informed deep neural network designed specifically to predict the temperature dependence of viscosity.

All three models were trained on data from the SciGlass database.



## GlassNet basic usage



Below is a minimal example of how to load and use the GlassNet model.



In [1]:
from glasspy.predict import GlassNet

model = GlassNet()
composition = "Li2O(SiO2)2"
predictions = model.predict(composition)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1612.586818,1286.966695,1187.265614,1070.224145,968.06795,901.811624,859.579252,825.204682,782.490883,759.993951,...,1717.653045,1040.284178,-6.241866,937.262437,788.40385,0.316586,0.3044,0.321539,0.318579,0.319566


A composition can also be defined using a dictionary.



In [2]:
composition = {
    "SiO2": 2,
    "Li2O": 1,
}
predictions = model.predict(composition)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1612.586818,1286.966695,1187.265614,1070.224145,968.06795,901.811624,859.579252,825.204682,782.490883,759.993951,...,1717.653045,1040.284178,-6.241866,937.262437,788.40385,0.316586,0.3044,0.321539,0.318579,0.319566


GlassNet also accepts `pandas` DataFrames as input. Note that each row represents a material and that only columns related to chemical composition can be present in the DataFrame.



In [3]:
import pandas as pd

data = [
    [1, 0, 2],
    [0, 1, 2],
    [1, 1, 2],
]

df = pd.DataFrame(data, columns=["Li2O", "Na2O", "SiO2"])
df

,Li2O,Na2O,SiO2
0,1,0,2
1,0,1,2
2,1,1,2


In [4]:
predictions = model.predict(df)

predictions

,T0,T1,T2,T3,T4,T5,T6,T7,T8,T9,...,Cp1673K,TMaxGrowthVelocity,MaxGrowthVelocity,CrystallizationPeak,CrystallizationOnset,SurfaceTensionAboveTg,SurfaceTension1173K,SurfaceTension1473K,SurfaceTension1573K,SurfaceTension1673K
0,1612.586818,1286.966759,1187.265614,1070.224201,968.067993,901.811569,859.579252,825.204682,782.490883,759.993951,...,1717.653120,1040.284178,-6.241865,937.262560,788.403850,0.316586,0.304400,0.321539,0.318579,0.319566
1,1700.171170,1521.813466,1287.576204,1130.354623,1036.780176,955.844950,911.027803,848.791035,810.406750,782.243953,...,1482.022135,1058.443194,-6.394961,963.434464,905.734703,0.279031,0.310337,0.290839,0.287974,0.272271
2,1457.627094,1246.016165,1114.455164,987.722016,934.740619,861.280868,794.972760,769.729834,730.195119,700.667308,...,1751.036502,1082.912718,-5.101020,875.732689,820.388535,0.299141,0.308964,0.319907,0.321466,0.294990


GlassNet can also predict viscosity and the MYEGA viscosity equation parameters.



In [5]:
predictions = model.predict_log10_viscosity(
    T=1000,
    composition=df,
)

predictions

array([3.53501871, 4.40572086, 2.83984945])

In [6]:
viscosity_parameters = model.viscosity_parameters(df)

viscosity_parameters

,log10_eta_infinity (Pa.s),Tg_MYEGA (K),fragility
0,-1.222603,701.790333,42.844319
1,-1.660096,710.677354,35.863589
2,-1.494931,647.762409,40.367431


## VITRIFY base usage



Below is a minimal example of how to load and use the VITRIFY model. Like GlassNet, VITRIFY accepts strings, dictionaries, and `pandas` DataFrames as input. It returns `1` if the composition is predicted to form a glass under typical laboratory conditions, or `0` otherwise.



In [7]:
from glasspy.predict import VITRIFY

model = VITRIFY()
predictions = model.predict(df)

predictions

array([1, 1, 1], dtype=int64)

You can also retrieve the probability that the composition will form a glass, as estimated by the model.



In [8]:
probabilities = model.predict_proba_glass(df)

probabilities

array([0.96919906, 0.99467885, 0.73446614])

## ViscNet basic usage



ViscNet is used similarly to GlassNet. However, since GlassNet outperforms ViscNet on viscosity prediction, using GlassNet is generally recommended for that task. The minimal example below demonstrates how to load and use ViscNet.



In [9]:
from glasspy.predict import ViscNet

model = ViscNet()
log10_viscosity = model.predict(T=1000, composition=df)

log10_viscosity

array([5.197888 , 5.6581345, 6.0721517], dtype=float32)

In [10]:
fragility = model.predict_fragility(df)

print(fragility)

[34.436737 29.760372 34.16933 ]


In [11]:
Tg = model.predict_Tg(df)

print(Tg)

[748.3823  739.4691  788.51385]
